In [1]:
import sys
from unittest.mock import MagicMock

# cspy C++ extension is not compatible with Python 3.14;
# since we use heuristic_only=True + cspy=False, we don't need it.
_mock_cspy = MagicMock()
_mock_cspy.REFCallback = type('REFCallback', (), {})
for _mod in ['cspy', 'cspy.algorithms', 'cspy.algorithms.bidirectional']:
    sys.modules[_mod] = _mock_cspy

import pandas as pd
from vrpy import VehicleRoutingProblem
import networkx as nx
from tqdm import tqdm 
import json
import warnings
import logging
import time
import numpy as np

warnings.filterwarnings("ignore")
logger = logging.getLogger()
logger.setLevel(logging.CRITICAL)


In [2]:
df_grids = pd.read_csv('../../results/utils/clustered_squares_all_clusters.csv').drop('Unnamed: 0', axis=1).drop(['maxLat','minLat','maxLong','minLong'], axis=1)
df_grids['Grid'] = df_grids['Grid'].apply(int)
df_grids = df_grids.astype({'candidate':'string', 'Grid': 'string'})

df_demands = pd.read_csv('../../results/utils/df_square_demand.csv').drop('Unnamed: 0', axis=1)
df_demands['Grid'] = df_demands['Grid'].apply(int)
df_demands = df_demands.astype({'Date':'string', 'Grid': 'string'})

df_dist_dcs = pd.read_csv('../../results/utils/df_dist_dcs.csv').drop('Unnamed: 0', axis=1)
df_dist_dcs['grid'] = df_dist_dcs['grid'].apply(int)
df_dist_dcs = df_dist_dcs.astype({'grid': 'string', 'CD': 'string'})

df_dist_grids = pd.read_csv('../../results/utils/df_dist_grids.csv').drop('Unnamed: 0', axis=1)
df_dist_grids['grid1'] = df_dist_grids['grid1'].apply(int)
df_dist_grids['grid2'] = df_dist_grids['grid2'].apply(int)
df_dist_grids = df_dist_grids.astype({'grid1': 'string', 'grid2': 'string'})

In [3]:
# Define iteration variables
num = df_grids['n_clusters'].sort_values(ascending=False).unique()
instances = df_demands.drop(['Grid', 'Date'], axis=1).columns
days = df_demands['Date'].unique()

# Define constraint variables
speed = {'car': 32, 'motorcycle': 40}
loading_time = 0.08
cost_per_km = {'car': 1.5, 'motorcycle': 1}
load_capacity=60 
duration=6
fixed_cost=80

In [4]:
def add_result(n, i, d, c, prob):
    
    result = {}
    result['n_clusters'] = [n]
    result['instance'] = [i]
    result['day'] = [d]
    result['cluster'] = [c]
    result['num_routes'] = len(prob.best_routes)
    result['avg_route_duration'] = sum(list(prob.best_routes_duration.values()))/len(prob.best_routes)
    result['total_route_cost'] = sum(list(prob.best_routes_cost.values()))
    result['max_route_cost'] = max(list(prob.best_routes_cost.values()))
    
    j=0
    total_cost_val = 0
    route_count = 0
    freight_per_route = []
    for route in list(prob.best_routes.values()):
        count = len(route) - 2
        route_cost = list(prob.best_routes_cost.values())[j]
        total_cost_val = total_cost_val + route_cost
        route_count = route_count + count
        freight_per_route.append(route_cost/count)
        j = j+1

    result['avg_freight'] = sum(freight_per_route)/len(freight_per_route)
    result['max_freight'] = max(freight_per_route)
    result['min_freight'] = min(freight_per_route)
    result['clients'] = route_count

    return result

In [ ]:
results = results_df

In [ ]:
results.to_csv('../../results/results_df.csv')